<a href="https://colab.research.google.com/github/Ud007it/Flyrank-ML-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ud007it/Flyrank-ML-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Fetch Hugging Face token securely from Colab secrets
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

# 2. Connect DuckDB and register HF Secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Base HF path
DATA_URL = "hf://datasets/FlyRank/internship-warehouse"
print("DuckDB connected and HF Secret registered successfully!")

DuckDB connected and HF Secret registered successfully!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row means: one pseudonymized content item (content_id) for one pseudonymized client (client_id), aggregated over one calendar month.

Table(s) used: fact_content_daily_performance, filtered to month=2026-03 (and month=2026-04 only for building the label).

Time window: March 2026 (2026-03) for all features; April 2026 (2026-04) used only to construct the forward-looking label.

What I'm predicting: is_refresh_candidate — whether a piece of content is likely to lose significant traffic the following month (a binary refresh/decay signal, not a continuous forecast).

One thing excluded: raw client URLs and query text — excluded to comply with FlyRank's pseudonymization agreement, and April's own metrics (m4_*) are excluded from the feature set since they're only allowed to build the label, not predict it.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Category	Field / Metric Name	Description / Source

Context	client_id, content_id, month	Primary keys and partition tracking.

Features	m3_clicks, m3_impressions, m3_ctr, m3_position_avg, m3_active_days	Monthly aggregates calculated strictly within 2026-03.

Label	is_refresh_candidate	Binary target (1 if m4_clicks < 0.7 * m3_clicks & m3_clicks ≥ 50; 0 otherwise).

Excluded	m4_clicks, m4_impressions, m4_ctr, client URLs, query text	Excluded from features to prevent data leakage and maintain pseudonymization safety.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
from sklearn.model_selection import train_test_split

q1 = f"""
SELECT client_hash_id, content_hash_id, COUNT(*) as row_count
FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_hash_id, content_hash_id
HAVING row_count > 1
LIMIT 10;
"""
duplicates = con.sql(q1).df()
print(f"Duplicate rows found at (client_hash_id, content_hash_id) grain for 2026-03: {len(duplicates)}")

q2 = f"""
SELECT
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(*) as total_daily_rows,
    COUNT(DISTINCT content_hash_id) as total_unique_content
FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
date_span_summary = con.sql(q2).df()
print(date_span_summary)

q3 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) as both_available_rows
FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
availability_summary = con.sql(q3).df()
print(availability_summary)

feature_query = f"""
WITH m3_data AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) as m3_clicks,
        SUM(gsc_impressions) as m3_impressions,
        AVG(COALESCE(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0), 0)) as m3_ctr,
        AVG(gsc_avg_position) as m3_position_avg,
        COUNT(DISTINCT report_date) as m3_active_days
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
m4_data AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) as m4_clicks
    FROM read_parquet('{DATA_URL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m3.client_hash_id, m3.content_hash_id,
    m3.m3_clicks, m3.m3_impressions, m3.m3_ctr, m3.m3_position_avg, m3.m3_active_days,
    m4.m4_clicks as LEAKED_m4_clicks,
    CASE WHEN m3.m3_clicks >= 50 AND COALESCE(m4.m4_clicks, 0) < (0.7 * m3.m3_clicks) THEN 1 ELSE 0 END as is_refresh_candidate
FROM m3_data m3
LEFT JOIN m4_data m4 ON m3.client_hash_id = m4.client_hash_id AND m3.content_hash_id = m4.content_hash_id;
"""
df = con.sql(feature_query).df().fillna(0)
print(f"Dataset shape: {df.shape}")
print(f"Positive labels (Refresh Candidates): {df['is_refresh_candidate'].sum()}")

# Split data to avoid overfitting evaluation
X = df[['m3_clicks','m3_impressions','m3_ctr','m3_position_avg','m3_active_days']]
y = df['is_refresh_candidate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Honest Model
clf_honest = RandomForestClassifier(random_state=42).fit(X_train, y_train)
y_prob = clf_honest.predict_proba(X_test)[:, 1]
score_honest = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC on Test Set (Honest): {score_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found at (client_hash_id, content_hash_id) grain for 2026-03: 10


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  total_daily_rows  total_unique_content
0 2026-03-01 2026-03-31           9841378                331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  both_available_rows
0     9841378             3611061               364347


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (176738, 9)
Positive labels (Refresh Candidates): 1117
ROC-AUC on Test Set (Honest): 0.9951


m3_clicks — knowable because it only sums clicks recorded within March 2026.
m3_impressions — knowable because it aggregates visibility metrics through March 31, 2026.
m3_ctr — knowable as the historical click/impression ratio through March 2026.
m3_position_avg — knowable as the average search rank observed during March 2026.
m3_active_days — knowable as a count of days with recorded activity, all before April 1, 2026.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced Client History: The panel is unbalanced because different clients have different gsc_data_start dates. A drop in traffic for a newer client may be due to incomplete historical indexing rather than actual content decay.

GSC vs. GA4 Window Discrepancy: Search Console (organic search visibility) and GA4 (user session engagement) integration start at different times. Relying purely on GSC fields ignores conversion intent, while filtering strictly for GA4 availability reduces row count.

Window Overlaps & Seasonality: Comparing adjacent 30-day windows (2026-03 vs. 2026-04) does not account for monthly seasonality or Google core algorithm updates occurring mid-month.


 4:Unbalanced Client History: The panel is unbalanced because different clients have different gsc_data_start dates. A drop in traffic for a newer client may be due to incomplete historical indexing rather than actual content decay.

 GSC vs. GA4 Window Discrepancy: Search Console (organic search visibility) and GA4 (user session engagement) integration start at different times. Relying purely on GSC fields ignores conversion intent, while filtering strictly for GA4 availability reduces row count.

 Window Overlaps & Seasonality: Comparing adjacent 30-day windows (2026-03 vs. 2026-04) does not account for monthly seasonality or Google core algorithm updates occurring mid-month.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.